In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# torch.set_default_device("cuda")

model = AutoModelForCausalLM.from_pretrained("microsoft/phi-1", torch_dtype="auto")
tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-1")

inputs = tokenizer('''def print_prime(n):
   """
   Print all primes between 1 and n
   """''', return_tensors="pt", return_attention_mask=False)

outputs = model.generate(**inputs, max_length=200)
text = tokenizer.batch_decode(outputs)[0]
print(text)


In [8]:
# !pip install --upgrade torch torchvision torchaudio
!pip install accelerate

In [ ]:
import os

# --- hard-disable CUDA in this process ---
os.environ.pop("PYTORCH_DEFAULT_DEVICE", None)  # if you (or a lib) set this to "cuda"
os.environ["CUDA_VISIBLE_DEVICES"] = ""         # hide GPUs even if present

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Pick device & dtype
if torch.backends.mps.is_available():
    torch.set_default_device("mps")
    dtype = torch.float16
    device = torch.device("mps")
else:
    torch.set_default_device("cpu")
    dtype = torch.float32
    device = torch.device("cpu")

print("Torch:", torch.__version__)
print("Default device:", torch.tensor([]).device)  # should be mps or cpu, NOT cuda

model_name = "microsoft/phi-1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype)
model.to(device).eval()

prompt = '''def print_prime(n):
   """
   Print all primes between 1 and n
   """'''

inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Torch: 2.8.0
Default device: mps:0


model.safetensors:   0%|          | 0.00/2.84G [00:00<?, ?B/s]